## Step 1: Document Ingestion

We load our two source PDFs — "Attention Is All You Need" and the original RAG paper —
and split them into overlapping chunks.

**Why PyPDFLoader:** LangChain's PyPDFLoader reads a PDF page-by-page and keeps track of
which page each piece of text came from. This page metadata is important later — it lets
us cite the exact source location when the chatbot answers a question.

**Why chunk_size=700, chunk_overlap=100:** Academic papers have dense, idea-per-paragraph
writing. 700 characters is roughly 150-200 words — enough to hold one full explanation
(e.g., one paragraph on self-attention) without blending multiple unrelated ideas into
the same chunk. The 100-character overlap means if an idea happens to fall right at a
chunk boundary, it still appears in full in at least one chunk, so we don't lose it to
an unlucky cut.

**Why RecursiveCharacterTextSplitter specifically:** Unlike a plain fixed-length splitter,
this one tries to break on paragraph breaks first, then sentences, then words — only
falling back to a hard character cut as a last resort. This keeps chunks more coherent.


In [ ]:
!pip install -q langchain-text-splitters

In [2]:
# Install required packages (only needs to run once per Colab session)
!pip install -q langchain langchain-community langchain-text-splitters pypdf

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Load both PDFs (update these paths if you uploaded to Drive instead of /content/)
transformer_loader = PyPDFLoader("/content/Attention is all you need.pdf")
rag_loader = PyPDFLoader("/content/Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks.pdf")

transformer_pages = transformer_loader.load()
rag_pages = rag_loader.load()

print(f"Transformer paper: {len(transformer_pages)} pages loaded")
print(f"RAG paper: {len(rag_pages)} pages loaded")

# Tag each page with which paper it came from, before we lose track after chunking
for page in transformer_pages:
    page.metadata["source_paper"] = "Attention Is All You Need"

for page in rag_pages:
    page.metadata["source_paper"] = "Retrieval-Augmented Generation (RAG)"

# Combine both documents into one list
all_pages = transformer_pages + rag_pages

# Split into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=100,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = text_splitter.split_documents(all_pages)

print(f"\nTotal chunks created: {len(chunks)}")
print(f"\nExample chunk:\n{'-'*50}")
print(chunks[0].page_content)
print(f"\nMetadata: {chunks[0].metadata}")

Transformer paper: 15 pages loaded
RAG paper: 19 pages loaded

Total chunks created: 198

Example chunk:
--------------------------------------------------
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗ †
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗ ‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent or

Metadata: {'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024

## Step 2: Embedding & Indexing

We convert each of our 198 chunks into a numerical vector (embedding) that captures
its meaning, then store all vectors in a FAISS index for fast similarity search.

**Why all-MiniLM-L6-v2:** This is a lightweight sentence-transformers model (~80MB)
that maps text to a 384-dimensional vector. It runs quickly on CPU — important since
Colab's free tier may not always give us a GPU — and is a widely-used, well-tested
baseline for semantic search tasks. Larger models like all-mpnet-base-v2 give a small
accuracy boost but are noticeably slower; for 198 chunks, that trade-off isn't worth it.

**Why FAISS:** FAISS (Facebook AI Similarity Search) is a free, local vector database
built specifically for fast nearest-neighbor search over embeddings. It requires no
server setup (unlike some hosted vector DBs) and runs entirely in-memory here, which
is perfect for a document set of this size.

**How the index works:** Each chunk's embedding gets stored alongside its original
text and metadata (page number, source paper). When a user asks a question, we embed
the question the same way, then ask FAISS: "which stored vectors are closest to this
one?" Those closest chunks are our retrieved context.


In [3]:
!pip install -q sentence-transformers faiss-cpu

from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# Load the embedding model
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

# Extract just the text content from each chunk (we'll keep chunks list for metadata lookup)
chunk_texts = [chunk.page_content for chunk in chunks]

print(f"Embedding {len(chunk_texts)} chunks... (this may take a minute on CPU)")

# Generate embeddings for all chunks at once (batched internally for efficiency)
chunk_embeddings = embedding_model.encode(
    chunk_texts,
    show_progress_bar=True,
    convert_to_numpy=True
)

print(f"\nEmbedding shape: {chunk_embeddings.shape}")
# Should print (198, 384) -> 198 chunks, each represented as a 384-number vector

# Build the FAISS index
embedding_dim = chunk_embeddings.shape[1]
index = faiss.IndexFlatL2(embedding_dim)  # L2 = Euclidean distance between vectors
index.add(chunk_embeddings)

print(f"FAISS index built with {index.ntotal} vectors")

# Quick sanity check: embed a test question and find its nearest chunk
test_question = "What is self-attention?"
test_embedding = embedding_model.encode([test_question], convert_to_numpy=True)

k = 3  # top-3 nearest chunks
distances, indices = index.search(test_embedding, k)

print(f"\nTest query: '{test_question}'")
print(f"Top {k} matching chunks:\n{'-'*50}")
for rank, idx in enumerate(indices[0]):
    print(f"\nRank {rank+1} (distance: {distances[0][rank]:.4f}):")
    print(f"Source: {chunks[idx].metadata['source_paper']}, Page {chunks[idx].metadata['page_label']}")
    print(chunks[idx].page_content[:200] + "...")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 70.5 MB/s eta 0:00:00


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding 198 chunks... (this may take a minute on CPU)


Batches:   0%|          | 0/7 [00:00<?, ?it/s]


Embedding shape: (198, 384)
FAISS index built with 198 vectors

Test query: 'What is self-attention?'
Top 3 matching chunks:
--------------------------------------------------

Rank 1 (distance: 0.9262):
Source: Attention Is All You Need, Page 2
in the distance between positions, linearly for ConvS2S and logarithmically for ByteNet. This makes
it more difficult to learn dependencies between distant positions [ 12]. In the Transformer this is
...

Rank 2 (distance: 1.0168):
Source: Attention Is All You Need, Page 14
Full attentions for head 5. Bottom: Isolated attentions from just the word ‘its’ for attention heads 5
and 6. Note that the attentions are very sharp for this word.
14...

Rank 3 (distance: 1.0349):
Source: Attention Is All You Need, Page 15
Figure 5: Many of the attention heads exhibit behaviour that seems related to the structure of the
sentence. We give two such examples above, from two different heads from the encoder self-attention
a...


In [4]:
def retrieve_chunks(query, k=4):
    """
    Given a user query, returns the top-k most relevant chunks
    along with their source metadata and similarity distance.
    """
    query_embedding = embedding_model.encode([query], convert_to_numpy=True)
    distances, indices = index.search(query_embedding, k)

    results = []
    for rank, idx in enumerate(indices[0]):
        results.append({
            "rank": rank + 1,
            "text": chunks[idx].page_content,
            "source_paper": chunks[idx].metadata["source_paper"],
            "page": chunks[idx].metadata["page_label"],
            "distance": float(distances[0][rank])
        })
    return results


def print_retrieval_results(query, k=4):
    """Helper to nicely print retrieval results for testing/debugging."""
    results = retrieve_chunks(query, k)
    print(f"Query: '{query}'\n{'='*60}")
    for r in results:
        print(f"\nRank {r['rank']} | {r['source_paper']} (Page {r['page']}) | distance: {r['distance']:.4f}")
        print(f"{r['text'][:250]}...")
    return results


# Test it with a couple of questions spanning both papers
print_retrieval_results("What is self-attention?")
print("\n\n" + "#"*60 + "\n")
print_retrieval_results("How does retrieval-augmented generation combine parametric and non-parametric memory?")

Query: 'What is self-attention?'

Rank 1 | Attention Is All You Need (Page 2) | distance: 0.9262
in the distance between positions, linearly for ConvS2S and logarithmically for ByteNet. This makes
it more difficult to learn dependencies between distant positions [ 12]. In the Transformer this is
reduced to a constant number of operations, albeit...

Rank 2 | Attention Is All You Need (Page 14) | distance: 1.0168
Full attentions for head 5. Bottom: Isolated attentions from just the word ‘its’ for attention heads 5
and 6. Note that the attentions are very sharp for this word.
14...

Rank 3 | Attention Is All You Need (Page 15) | distance: 1.0349
Figure 5: Many of the attention heads exhibit behaviour that seems related to the structure of the
sentence. We give two such examples above, from two different heads from the encoder self-attention
at layer 5 of 6. The heads clearly learned to perfo...

Rank 4 | Attention Is All You Need (Page 6) | distance: 1.0355
during training.
4 Why Self-At

[{'rank': 1,
  'text': 'edge is still limited, and hence on knowledge-intensive tasks, their performance\nlags behind task-speciﬁc architectures. Additionally, providing provenance for their\ndecisions and updating their world knowledge remain open research problems. Pre-\ntrained models with a differentiable access mechanism to explicit non-parametric\nmemory have so far been only investigated for extractive downstream tasks. We\nexplore a general-purpose ﬁne-tuning recipe for retrieval-augmented generation\n(RAG) — models which combine pre-trained parametric and non-parametric mem-\nory for language generation. We introduce RAG models where the parametric',
  'source_paper': 'Retrieval-Augmented Generation (RAG)',
  'page': '1',
  'distance': 0.5172176957130432},
 {'rank': 2,
  'text': 'rather than related training pairs. This said, RAG techniques may work well in these settings, and\ncould represent promising future work.\n6 Discussion\nIn this work, we presented hybrid generation m

## Step 4: Generation

We take the top-k retrieved chunks from Step 3, combine them with the user's
question into a prompt, and send that prompt to an LLM via the Groq API to
generate a grounded answer.

**Why Groq:** Groq offers a free-tier API that runs open models (like Llama 3)
at very high inference speed, with no cost for moderate usage — ideal for a
student project like this compared to paid APIs.

**Why Llama 3.1 8B (via Groq):** It's fast, free-tier friendly, and capable
enough to follow instructions like "only answer from the provided context" —
which is the core requirement of this task. We don't need a huge model since
we're not asking it to reason from scratch; we're asking it to summarize and
answer strictly from text we hand it.

**Why we explicitly instruct the model to only use provided context:** This
is the entire point of RAG — without this instruction, an LLM will happily
fall back on its own training data and hallucinate an answer that sounds
plausible but isn't actually grounded in our documents. The system prompt
below explicitly forbids that.

In [5]:
!pip install -q groq

from google.colab import userdata
from groq import Groq

# Retrieve the API key from Colab Secrets (never hardcoded in the notebook)
groq_api_key = userdata.get('GROQ_API_KEY')
client = Groq(api_key=groq_api_key)


def generate_answer(query, k=4):
    """
    Retrieves relevant chunks for the query, then asks the LLM to answer
    strictly based on those chunks. Returns both the answer and the
    source chunks used, for citation purposes.
    """
    retrieved = retrieve_chunks(query, k=k)

    # Build context block from retrieved chunks, each labeled with its source
    context_block = "\n\n".join([
        f"[Source: {r['source_paper']}, Page {r['page']}]\n{r['text']}"
        for r in retrieved
    ])

    system_prompt = (
        "You are a document Q&A assistant. Answer the user's question using "
        "ONLY the context provided below. Do not use any outside knowledge, "
        "even if you know the answer from general training. "
        "If the answer is not contained in the provided context, respond "
        "exactly with: 'I cannot answer this from the provided documents.' "
        "Do not guess, speculate, or fill in gaps with information not present "
        "in the context."
    )

    user_prompt = f"Context:\n{context_block}\n\nQuestion: {query}"

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.2,  # low temperature = more literal/grounded, less creative
    )

    answer = response.choices[0].message.content

    return {
        "answer": answer,
        "sources": retrieved
    }


def print_answer(query, k=4):
    """Helper to nicely display an answer with its sources."""
    result = generate_answer(query, k=k)
    print(f"Question: {query}\n{'='*60}")
    print(f"\nAnswer:\n{result['answer']}")
    print(f"\n{'-'*60}\nSources used:")
    for r in result['sources']:
        print(f"  - {r['source_paper']}, Page {r['page']} (distance: {r['distance']:.4f})")
    return result


# Test with one in-scope question
print_answer("What is self-attention?")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 2.1 MB/s eta 0:00:00
Question: What is self-attention?

Answer:
Self-attention, sometimes called intra-attention, is an attention mechanism relating different positions of a single sequence in order to compute a representation of the sequence.

------------------------------------------------------------
Sources used:
  - Attention Is All You Need, Page 2 (distance: 0.9262)
  - Attention Is All You Need, Page 14 (distance: 1.0168)
  - Attention Is All You Need, Page 15 (distance: 1.0349)
  - Attention Is All You Need, Page 6 (distance: 1.0355)


{'answer': 'Self-attention, sometimes called intra-attention, is an attention mechanism relating different positions of a single sequence in order to compute a representation of the sequence.',
 'sources': [{'rank': 1,
   'text': 'in the distance between positions, linearly for ConvS2S and logarithmically for ByteNet. This makes\nit more difficult to learn dependencies between distant positions [ 12]. In the Transformer this is\nreduced to a constant number of operations, albeit at the cost of reduced effective resolution due\nto averaging attention-weighted positions, an effect we counteract with Multi-Head Attention as\ndescribed in section 3.2.\nSelf-attention, sometimes called intra-attention is an attention mechanism relating different positions\nof a single sequence in order to compute a representation of the sequence. Self-attention has been',
   'source_paper': 'Attention Is All You Need',
   'page': '2',
   'distance': 0.9261771440505981},
  {'rank': 2,
   'text': 'Full attent

In [6]:
# Test with an out-of-scope question (something NOT covered by either paper)
print_answer("What is the capital of France?")


Question: What is the capital of France?

Answer:
I cannot answer this from the provided documents.

------------------------------------------------------------
Sources used:
  - Attention Is All You Need, Page 14 (distance: 1.7410)
  - Attention Is All You Need, Page 3 (distance: 1.7467)
  - Attention Is All You Need, Page 6 (distance: 1.7489)
  - Attention Is All You Need, Page 5 (distance: 1.7523)


{'answer': 'I cannot answer this from the provided documents.',
 'sources': [{'rank': 1,
   'text': 'Full attentions for head 5. Bottom: Isolated attentions from just the word ‘its’ for attention heads 5\nand 6. Note that the attentions are very sharp for this word.\n14',
   'source_paper': 'Attention Is All You Need',
   'page': '14',
   'distance': 1.7409553527832031},
  {'rank': 2,
   'text': 'the two sub-layers, followed by layer normalization [ 1]. That is, the output of each sub-layer is\nLayerNorm(x + Sublayer(x)), where Sublayer(x) is the function implemented by the sub-layer\nitself. To facilitate these residual connections, all sub-layers in the model, as well as the embedding\nlayers, produce outputs of dimension dmodel = 512.\nDecoder: The decoder is also composed of a stack of N = 6identical layers. In addition to the two\nsub-layers in each encoder layer, the decoder inserts a third sub-layer, which performs multi-head\nattention over the output of the encoder stack. Simi

In [7]:
print_answer("What was the exact GPU training cost in dollars for the RAG paper's experiments?")

Question: What was the exact GPU training cost in dollars for the RAG paper's experiments?

Answer:
I cannot answer this from the provided documents.

------------------------------------------------------------
Sources used:
  - Attention Is All You Need, Page 7 (distance: 1.1697)
  - Retrieval-Augmented Generation (RAG), Page 17 (distance: 1.1841)
  - Attention Is All You Need, Page 8 (distance: 1.2106)
  - Retrieval-Augmented Generation (RAG), Page 18 (distance: 1.3583)


{'answer': 'I cannot answer this from the provided documents.',
 'sources': [{'rank': 1,
   'text': 'We trained our models on one machine with 8 NVIDIA P100 GPUs. For our base models using\nthe hyperparameters described throughout the paper, each training step took about 0.4 seconds. We\ntrained the base models for a total of 100,000 steps or 12 hours. For our big models,(described on the\nbottom line of table 3), step time was 1.0 seconds. The big models were trained for 300,000 steps\n(3.5 days).\n5.3 Optimizer\nWe used the Adam optimizer [20] with β1 = 0.9, β2 = 0.98 and ϵ = 10−9. We varied the learning\nrate over the course of training, according to the formula:\nlrate = d−0.5\nmodel · min(step_num−0.5, step_num · warmup_steps−1.5) (3)',
   'source_paper': 'Attention Is All You Need',
   'page': '7',
   'distance': 1.1696819067001343},
  {'rank': 2,
   'text': 'tions and worked examples in a full instructions tab. We included some gold sentences in order to\nassess the accuracy of 

## Step 5: Grounding Check

Relying only on prompt instructions ("only use the provided context") is fragile —
LLMs sometimes deviate from instructions, especially with ambiguous questions.
To make grounding more robust, we add two additional safeguards on top of the
prompt instruction:

1. **Distance threshold check:** If the retrieved chunks' similarity distances
   are all above a certain threshold, that means even our "best" matches aren't
   actually very similar to the question — a strong signal the documents likely
   don't cover this topic at all. In that case, we skip calling the LLM entirely
   and return the refusal message directly. This also saves an API call.

2. **Refusal phrase detection:** We check if the LLM's answer contains our
   expected refusal phrase, so downstream code (like the Streamlit UI) can
   reliably detect "no answer found" cases and, for example, style them
   differently (e.g. a warning icon) versus a real grounded answer.

**Why a distance threshold and not just trusting the LLM:** Looking at our own
test results, in-scope questions retrieved chunks with distances around 0.5–1.0,
while our clearly out-of-scope GPU-cost question still retrieved distances in a
similar range (1.17–1.36) because retrieval always returns *something* — FAISS
finds the

In [8]:
REFUSAL_MESSAGE = "I cannot answer this from the provided documents."
DISTANCE_THRESHOLD = 1.5  # tune this based on your own testing


def generate_answer_with_grounding_check(query, k=4):
    """
    Same as generate_answer, but adds a distance-based pre-filter:
    if even the closest retrieved chunk is too far (semantically unrelated),
    skip the LLM call entirely and return the refusal directly.
    """
    retrieved = retrieve_chunks(query, k=k)

    best_distance = retrieved[0]["distance"]

    if best_distance > DISTANCE_THRESHOLD:
        return {
            "answer": REFUSAL_MESSAGE,
            "sources": retrieved,
            "grounded": False,
            "skipped_llm_call": True
        }

    result = generate_answer(query, k=k)
    is_grounded = REFUSAL_MESSAGE.lower() not in result["answer"].lower()

    return {
        "answer": result["answer"],
        "sources": result["sources"],
        "grounded": is_grounded,
        "skipped_llm_call": False
    }


def print_grounded_answer(query, k=4):
    result = generate_answer_with_grounding_check(query, k=k)
    print(f"Question: {query}\n{'='*60}")
    print(f"\nAnswer:\n{result['answer']}")
    print(f"\nGrounded: {result['grounded']} | LLM call skipped: {result['skipped_llm_call']}")
    print(f"\n{'-'*60}\nSources retrieved:")
    for r in result['sources']:
        print(f"  - {r['source_paper']}, Page {r['page']} (distance: {r['distance']:.4f})")
    return result


# Re-test all three questions with the new grounding-checked function
print_grounded_answer("What is self-attention?")
print("\n" + "#"*60 + "\n")
print_grounded_answer("What is the capital of France?")
print("\n" + "#"*60 + "\n")
print_grounded_answer("What was the exact GPU training cost in dollars for the RAG paper's experiments?")

Question: What is self-attention?

Answer:
Self-attention, sometimes called intra-attention, is an attention mechanism relating different positions of a single sequence in order to compute a representation of the sequence.

Grounded: True | LLM call skipped: False

------------------------------------------------------------
Sources retrieved:
  - Attention Is All You Need, Page 2 (distance: 0.9262)
  - Attention Is All You Need, Page 14 (distance: 1.0168)
  - Attention Is All You Need, Page 15 (distance: 1.0349)
  - Attention Is All You Need, Page 6 (distance: 1.0355)

############################################################

Question: What is the capital of France?

Answer:
I cannot answer this from the provided documents.

Grounded: False | LLM call skipped: True

------------------------------------------------------------
Sources retrieved:
  - Attention Is All You Need, Page 14 (distance: 1.7410)
  - Attention Is All You Need, Page 3 (distance: 1.7467)
  - Attention Is All Yo

{'answer': 'I cannot answer this from the provided documents.',
 'sources': [{'rank': 1,
   'text': 'We trained our models on one machine with 8 NVIDIA P100 GPUs. For our base models using\nthe hyperparameters described throughout the paper, each training step took about 0.4 seconds. We\ntrained the base models for a total of 100,000 steps or 12 hours. For our big models,(described on the\nbottom line of table 3), step time was 1.0 seconds. The big models were trained for 300,000 steps\n(3.5 days).\n5.3 Optimizer\nWe used the Adam optimizer [20] with β1 = 0.9, β2 = 0.98 and ϵ = 10−9. We varied the learning\nrate over the course of training, according to the formula:\nlrate = d−0.5\nmodel · min(step_num−0.5, step_num · warmup_steps−1.5) (3)',
   'source_paper': 'Attention Is All You Need',
   'page': '7',
   'distance': 1.1696819067001343},
  {'rank': 2,
   'text': 'tions and worked examples in a full instructions tab. We included some gold sentences in order to\nassess the accuracy of 

## Step 5b: Persisting the Index for Streamlit

The Streamlit app runs as a separate process outside this notebook, so it needs
its own saved copy of the FAISS index and the chunk data — otherwise it would
have to re-download the embedding model and re-embed all 198 chunks every time
someone launches the app, which is slow and unnecessary.

**Why FAISS's native write_index:** FAISS provides a built-in method to
serialize its index to a single binary file, which is the most reliable way
to persist it exactly as-is.

**Why pickle for the chunks:** Our chunks are LangChain Document objects
containing both text and metadata (page number, source paper). Pickle can
serialize this structure directly without us needing to manually convert
it to a different format like JSON.

In [9]:
import pickle
import faiss

# Save the FAISS index to disk
faiss.write_index(index, "faiss_index.index")

# Save the chunks (text + metadata) to disk using pickle
with open("chunks.pkl", "wb") as f:
    pickle.dump(chunks, f)

print("Saved: faiss_index.index and chunks.pkl")
print(f"Index contains {index.ntotal} vectors")
print(f"Chunks list contains {len(chunks)} items")

Saved: faiss_index.index and chunks.pkl
Index contains 198 vectors
Chunks list contains 198 items
